# Multi-Agent Loan Underwriting System

## CSE476 – CA2 Project 2

### Project Overview

This project implements a multi-agent loan underwriting
system using Microsoft Agent Framework and Python.

The system consists of five agents:

1. Document Agent
2. EMI Computation Agent
3. Affordability Agent
4. Credit-History Agent
5. Underwriting Manager

### Objectives

- Automate loan application processing.
- Extract and validate applicant information.
- Calculate monthly loan EMI.
- Evaluate applicant affordability.
- Analyze credit history.
- Generate a final underwriting decision.

### Architecture

Manager-Worker Multi-Agent Architecture

### Technologies

- Python 3.11
- Microsoft Agent Framework
- OpenAI API
- Shared Applicant Memory
- JSON Credit Database
- Jupyter Notebook

In [2]:
import sys
import os
import json

from shared_memory import shared_memory

from agents.document_agent import (
    extract_and_store_application,
    validate_stored_application
)

from agents.emi_agent import (
    compute_and_store_emi,
    create_repayment_summary
)

from agents.affordability_agent import (
    calculate_and_store_affordability,
    evaluate_and_store_affordability
)

from agents.credit_agent import (
    lookup_and_store_credit_history,
    evaluate_and_store_credit_risk
)

from offline_workflow import run_offline_underwriting

print("All project modules imported successfully!")

All project modules imported successfully!


## System Architecture

The system follows a Manager-Worker architecture.

The Underwriting Manager coordinates four specialist agents.
All agents communicate through shared applicant memory.

The final decision can be:

- SANCTION
- DECLINE
- REFER

In [4]:
# DOCUMENT AGENT DEMONSTRATION

applicant_id = "APP001"

# Create applicant record
shared_memory.create_applicant(applicant_id)

# Sample loan application
application_text = """
Applicant Name: Rahul Sharma
Monthly Income: 65000
Existing EMI: 5000
Requested Loan: 800000
Annual Interest Rate: 10
Tenure Months: 60
"""

# Extract application details
extraction_result = extract_and_store_application(
    applicant_id,
    application_text
)

print("DOCUMENT EXTRACTION RESULT")
print(extraction_result)

# Validate application
validation_result = validate_stored_application(
    applicant_id
)

print("\nDOCUMENT VALIDATION RESULT")
print(validation_result)

print("\nDOCUMENT AGENT DEMONSTRATION COMPLETED")

DOCUMENT EXTRACTION RESULT
{"success": true, "applicant_id": "APP001", "extracted_data": {"applicant_name": "Rahul Sharma", "monthly_income": 65000.0, "existing_emi": 5000.0, "loan_amount": 800000.0, "annual_interest_rate": 10.0, "tenure_months": 60.0}}

DOCUMENT VALIDATION RESULT
{"applicant_id": "APP001", "validation": {"valid": true, "missing_fields": [], "errors": [], "income_missing": false}}

DOCUMENT AGENT DEMONSTRATION COMPLETED


In [5]:
# SHARED APPLICANT MEMORY

applicant_record = shared_memory.get_record(applicant_id)

print(json.dumps(applicant_record, indent=4, default=str))

{
    "applicant_id": "APP001",
    "application": {
        "applicant_name": "Rahul Sharma",
        "monthly_income": 65000.0,
        "existing_emi": 5000.0,
        "loan_amount": 800000.0,
        "annual_interest_rate": 10.0,
        "tenure_months": 60.0
    },
    "document_analysis": {
        "extraction_status": "COMPLETED",
        "extracted_fields": [
            "applicant_name",
            "monthly_income",
            "existing_emi",
            "loan_amount",
            "annual_interest_rate",
            "tenure_months"
        ],
        "validation_status": "VALID",
        "missing_fields": [],
        "validation_errors": [],
        "income_missing": false
    },
    "emi_analysis": {},
    "affordability_analysis": {},
    "credit_analysis": {},
    "final_decision": {},
    "execution_trace": [
        {
            "timestamp": "2026-11-15T09:46:06.214453+00:00",
            "agent": "Shared Memory",
            "message": "Applicant record created"
      

In [6]:
# EMI COMPUTATION AGENT DEMONSTRATION

emi_result = compute_and_store_emi(applicant_id)

repayment_result = create_repayment_summary(applicant_id)

print("EMI COMPUTATION RESULT")
print(json.dumps(emi_result, indent=4))

print("\nREPAYMENT SUMMARY")
print(json.dumps(repayment_result, indent=4))

EMI COMPUTATION RESULT
{
    "monthly_emi": 16997.64,
    "principal": 800000.0,
    "annual_interest_rate": 10.0,
    "tenure_months": 60,
    "calculation_method": "Reducing Balance"
}

REPAYMENT SUMMARY
{
    "total_repayment": 1019858.4,
    "total_interest": 219858.4,
    "principal": 800000.0,
    "tenure_months": 60
}


In [7]:
# AFFORDABILITY AGENT DEMONSTRATION

obligation_result = calculate_and_store_affordability(
    applicant_id
)

affordability_result = evaluate_and_store_affordability(
    applicant_id
)

print("OBLIGATION RATIO")
print(json.dumps(obligation_result, indent=4))

print("\nAFFORDABILITY EVALUATION")
print(json.dumps(affordability_result, indent=4))

OBLIGATION RATIO
{
    "monthly_income": 65000.0,
    "existing_emi": 5000.0,
    "proposed_emi": 16997.64,
    "total_monthly_obligations": 21997.64,
    "obligation_ratio": 33.84
}

AFFORDABILITY EVALUATION
{
    "affordable": true,
    "status": "PASS",
    "obligation_ratio": 33.84,
    "maximum_allowed_ratio": 40.0,
    "reason": "Monthly obligations are within the configured limit."
}


In [8]:
# CREDIT-HISTORY AGENT DEMONSTRATION

credit_history = lookup_and_store_credit_history(
    applicant_id
)

credit_result = evaluate_and_store_credit_risk(
    applicant_id
)

print("CREDIT HISTORY")
print(json.dumps(credit_history, indent=4))

print("\nCREDIT RISK EVALUATION")
print(json.dumps(credit_result, indent=4))

CREDIT HISTORY
{
    "found": true,
    "status": "FOUND",
    "history": {
        "previous_defaults": 0,
        "late_payments": 0,
        "active_overdue": 0
    }
}

CREDIT RISK EVALUATION
{
    "status": "NO_RECORDED_FLAGS",
    "manual_review_required": false,
    "red_flags": [],
    "reason": "No adverse indicators found in available records."
}


In [9]:
# COMPLETE APPLICANT RECORD

final_record = shared_memory.get_record(applicant_id)

print(json.dumps(final_record, indent=4, default=str))

{
    "applicant_id": "APP001",
    "application": {
        "applicant_name": "Rahul Sharma",
        "monthly_income": 65000.0,
        "existing_emi": 5000.0,
        "loan_amount": 800000.0,
        "annual_interest_rate": 10.0,
        "tenure_months": 60.0
    },
    "document_analysis": {
        "extraction_status": "COMPLETED",
        "extracted_fields": [
            "applicant_name",
            "monthly_income",
            "existing_emi",
            "loan_amount",
            "annual_interest_rate",
            "tenure_months"
        ],
        "validation_status": "VALID",
        "missing_fields": [],
        "validation_errors": [],
        "income_missing": false
    },
    "emi_analysis": {
        "monthly_emi": 16997.64,
        "principal": 800000.0,
        "annual_interest_rate": 10.0,
        "tenure_months": 60,
        "calculation_method": "Reducing Balance",
        "status": "COMPLETED",
        "repayment_summary": {
            "total_repayment": 10198

## Underwriting Manager Demonstration

The Underwriting Manager coordinates the four specialist
agents using a manager-worker workflow.

It collects their results, applies the underwriting rules,
and saves the final decision in shared applicant memory.

Offline mode uses deterministic Python orchestration.

In [11]:
# UNDERWRITING MANAGER DEMONSTRATION

manager_result = run_offline_underwriting(
    applicant_id,
    application_text
)

print("\nFINAL UNDERWRITING REPORT")
print(json.dumps(manager_result, indent=4, default=str))


OFFLINE LOAN UNDERWRITING
Applicant: APP001

STEP 1: Reading applicant record

STEP 2: Document Agent
Extraction: {"success": true, "applicant_id": "APP001", "extracted_data": {"applicant_name": "Rahul Sharma", "monthly_income": 65000.0, "existing_emi": 5000.0, "loan_amount": 800000.0, "annual_interest_rate": 10.0, "tenure_months": 60.0}}
Validation: {'valid': True, 'missing_fields': [], 'errors': [], 'income_missing': False}

STEP 3: EMI Computation Agent
EMI: {'monthly_emi': 16997.64, 'principal': 800000.0, 'annual_interest_rate': 10.0, 'tenure_months': 60, 'calculation_method': 'Reducing Balance'}
Repayment: {'total_repayment': 1019858.4, 'total_interest': 219858.4, 'principal': 800000.0, 'tenure_months': 60}

STEP 4: Affordability Agent
Obligation: {'monthly_income': 65000.0, 'existing_emi': 5000.0, 'proposed_emi': 16997.64, 'total_monthly_obligations': 21997.64, 'obligation_ratio': 33.84}
Affordability: {'affordable': True, 'status': 'PASS', 'obligation_ratio': 33.84, 'maximum_al